# Lesson 5a: n-Step Methods - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand n-Step Returns** - Bridge between TD and MC
2. **Master n-Step TD Prediction** - Multi-step bootstrapping
3. **Learn n-Step SARSA** - n-step on-policy control
4. **Learn n-Step Q-Learning** - n-step off-policy control
5. **Understand Eligibility Traces** - Backward view of n-step
6. **Master TD(λ)** - Continuous spectrum of methods
7. **Compare All Variants** - Trade-offs and performance

n-Step methods provide a unified framework connecting one-step TD and full Monte Carlo methods.

---

## Table of Contents

1. [Introduction to n-Step Methods](#intro)
2. [n-Step TD Prediction](#nstep-td)
3. [n-Step SARSA](#nstep-sarsa)
4. [n-Step Q-Learning](#nstep-q)
5. [Eligibility Traces](#traces)
6. [TD(λ) - Forward View](#td-lambda-forward)
7. [TD(λ) - Backward View](#td-lambda-backward)
8. [Choosing n and λ](#choosing)
9. [Theory and Convergence](#convergence)

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. Introduction to n-Step Methods <a id='intro'></a>

### The Spectrum of Methods

**Key insight**: TD and MC are extremes of a spectrum.

**One-step TD(0)**:
$$G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$$

**Two-step TD**:
$$G_t^{(2)} = R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+2})$$

**Three-step TD**:
$$G_t^{(3)} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \gamma^3 V(S_{t+3})$$

**n-step TD**:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

**Monte Carlo** ($n = \infty$):
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... + \gamma^{T-t-1} R_T$$

### Bias-Variance Trade-off

**Small n** (e.g., n=1):
- ✅ Low variance (few random steps)
- ❌ High bias (bootstrap error)
- ✅ Fast updates

**Large n** (approaching ∞):
- ✅ Low bias (true returns)
- ❌ High variance (many random steps)
- ❌ Slow updates (wait n steps)

**Intermediate n** (e.g., n=3-5):
- ⚖️ Balance bias and variance
- Often optimal in practice

### Why n-Step Methods?

1. **Faster learning**: More information per update than 1-step
2. **More stable**: Less variance than full MC
3. **Flexible**: Can tune n for problem
4. **Bridge to eligibility traces**: Foundation for TD(λ)

### Backup Diagrams

**1-step backup**:
```
S_t → S_{t+1}
```

**3-step backup**:
```
S_t → S_{t+1} → S_{t+2} → S_{t+3}
```

**Full MC backup**:
```
S_t → S_{t+1} → S_{t+2} → ... → S_T
```

## 2. n-Step TD Prediction <a id='nstep-td'></a>

### n-Step Return

**Definition**: The n-step return from time t is:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

More compactly:

$$G_t^{(n)} = \sum_{i=0}^{n-1} \gamma^i R_{t+i+1} + \gamma^n V(S_{t+n})$$

**For terminal states**: If episode terminates before n steps:

$$G_t^{(n)} = G_t \text{ for } t + n \geq T$$

### n-Step TD Update

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^{(n)} - V(S_t)]$$

**Key property**: Update happens n steps later (must wait for $R_{t+n}$ and $V(S_{t+n})$)

### Algorithm: n-Step TD Prediction

```
Input: policy π, step size α, n
Initialize V(s) arbitrarily for all s

For each episode:
  Initialize S_0
  Store state sequence: S_0, S_1, S_2, ...
  Store reward sequence: R_1, R_2, R_3, ...
  T ← ∞
  
  For t = 0, 1, 2, ...:
    If t < T:
      Take action A_t ~ π(·|S_t)
      Observe R_{t+1}, S_{t+1}
      If S_{t+1} is terminal: T ← t + 1
    
    τ ← t - n + 1  (time whose estimate is being updated)
    
    If τ ≥ 0:
      G ← Σ_{i=τ+1}^{min(τ+n,T)} γ^{i-τ-1} R_i
      If τ + n < T: G ← G + γ^n V(S_{τ+n})
      V(S_τ) ← V(S_τ) + α[G - V(S_τ)]
    
    If τ = T - 1: break
```

### Example: 3-Step TD

Episode: $S_0, R_1, S_1, R_2, S_2, R_3, S_3, R_4, S_4$ (terminal)

**Updates**:
- At $t=3$: Update $V(S_0)$ using $G_0^{(3)} = R_1 + \gamma R_2 + \gamma^2 R_3 + \gamma^3 V(S_3)$
- At $t=4$: Update $V(S_1)$ using $G_1^{(3)} = R_2 + \gamma R_3 + \gamma^2 R_4 + 0$ (terminal)
- At $t=4$: Update $V(S_2)$ using $G_2^{(2)} = R_3 + \gamma R_4$
- At $t=4$: Update $V(S_3)$ using $G_3^{(1)} = R_4$

**Notice**: Earlier states use full n-step return, later states use fewer steps (episode terminates)

### n-Step TD Error

**TD error** for n-step return:

$$\delta_t^{(n)} = G_t^{(n)} - V(S_t)$$

**Update**:
$$V(S_t) \leftarrow V(S_t) + \alpha \delta_t^{(n)}$$

## 3. n-Step SARSA <a id='nstep-sarsa'></a>

### n-Step Action-Value Return

For control, we need action values:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n Q(S_{t+n}, A_{t+n})$$

With termination handling:

$$G_t^{(n)} = \sum_{i=0}^{\min(n-1, T-t-1)} \gamma^i R_{t+i+1} + \begin{cases}
\gamma^n Q(S_{t+n}, A_{t+n}) & \text{if } t+n < T \\
0 & \text{if } t+n \geq T
\end{cases}$$

### n-Step SARSA Update

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[G_t^{(n)} - Q(S_t, A_t)]$$

### Algorithm: n-Step SARSA

```
Initialize Q(s,a) arbitrarily for all s, a
Parameters: α, γ, ε, n

For each episode:
  Initialize S_0
  Choose A_0 ~ ε-greedy(Q(S_0, ·))
  Store sequences: (S_0, A_0), (S_1, A_1), ...
  Store rewards: R_1, R_2, ...
  T ← ∞
  
  For t = 0, 1, 2, ...:
    If t < T:
      Take A_t, observe R_{t+1}, S_{t+1}
      If S_{t+1} terminal: T ← t + 1
      Else: Choose A_{t+1} ~ ε-greedy(Q(S_{t+1}, ·))
    
    τ ← t - n + 1
    
    If τ ≥ 0:
      G ← Σ_{i=τ+1}^{min(τ+n,T)} γ^{i-τ-1} R_i
      If τ + n < T: G ← G + γ^n Q(S_{τ+n}, A_{τ+n})
      Q(S_τ, A_τ) ← Q(S_τ, A_τ) + α[G - Q(S_τ, A_τ)]
    
    If τ = T - 1: break
```

### Comparison with 1-Step SARSA

**1-step SARSA**:
- Update: $Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$
- Updates immediately at each step

**n-step SARSA**:
- Update: Uses $R_{t+1}, ..., R_{t+n}$ and $Q(S_{t+n}, A_{t+n})$
- Updates delayed by n steps
- More stable (averages over n steps)

### On-Policy Nature

n-Step SARSA is **on-policy** because:
- All actions $A_t, A_{t+1}, ..., A_{t+n}$ come from same ε-greedy policy
- Learns value of policy being followed

## 4. n-Step Q-Learning <a id='nstep-q'></a>

### Off-Policy n-Step

**Challenge**: Q-Learning is off-policy (learns greedy while acting ε-greedy)

**Two approaches**:
1. n-Step Tree Backup
2. n-Step Q(σ)

### n-Step Tree Backup

**Idea**: Use expectation instead of sampling for non-selected actions

**1-step tree backup**:
$$G_t^{(1)} = R_{t+1} + \gamma \sum_a \pi(a|S_{t+1}) Q(S_{t+1}, a)$$

**2-step tree backup**:
$$G_t^{(2)} = R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1}) Q(S_{t+1}, a) + \gamma \pi(A_{t+1}|S_{t+1}) [R_{t+2} + \gamma \sum_a \pi(a|S_{t+2}) Q(S_{t+2}, a)]$$

**General n-step tree backup**:
- For taken actions: Use actual reward and recurse
- For non-taken actions: Use expected value

**Update**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[G_t - Q(S_t, A_t)]$$

### n-Step Expected SARSA

Simpler off-policy approach:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n \sum_a \pi(a|S_{t+n}) Q(S_{t+n}, a)$$

**Special case** (greedy π):
$$\sum_a \pi(a|S_{t+n}) Q(S_{t+n}, a) = \max_a Q(S_{t+n}, a)$$

This gives **n-step Q-learning**.

### Importance Sampling for n-Step

**Alternative**: Use importance sampling ratio

$$\rho_{t:t+n-1} = \prod_{k=t}^{\min(t+n-1, T-1)} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

**Update**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \rho_{t:t+n-1} [G_t^{(n)} - Q(S_t, A_t)]$$

**Problem**: High variance for large n

**Solution**: Tree backup (no importance sampling needed)

## 5. Eligibility Traces <a id='traces'></a>

### The Credit Assignment Problem

**Question**: Which past states/actions should get credit for current reward?

**Answer**: Eligibility traces track "eligibility" for updates

### Eligibility Trace Definition

For each state s, maintain trace $z(s)$:

**Accumulating traces**:
$$z_t(s) = \begin{cases}
\gamma \lambda z_{t-1}(s) + 1 & \text{if } s = S_t \\
\gamma \lambda z_{t-1}(s) & \text{if } s \neq S_t
\end{cases}$$

**Replacing traces**:
$$z_t(s) = \begin{cases}
1 & \text{if } s = S_t \\
\gamma \lambda z_{t-1}(s) & \text{if } s \neq S_t
\end{cases}$$

**Parameters**:
- $\gamma$: Discount factor
- $\lambda \in [0,1]$: Trace decay rate

### Intuition

- **z(s) = 0**: State s not recently visited, no credit
- **z(s) = 1**: State s just visited, full credit  
- **z(s) ∈ (0,1)**: State s visited in past, partial credit

**Decay**: $\lambda$ controls how fast eligibility fades
- $\lambda = 0$: Only current state eligible (1-step TD)
- $\lambda = 1$: All visited states remain eligible (MC-like)
- $\lambda \in (0,1)$: Intermediate (n-step-like)

### Trace Dynamics

Episode: Visit states A, B, C, B, D

**Accumulating traces** ($\gamma=1, \lambda=0.9$):

| Time | Visited | z(A) | z(B) | z(C) | z(D) |
|------|---------|------|------|------|------|
| 0    | A       | 1.0  | 0    | 0    | 0    |
| 1    | B       | 0.9  | 1.0  | 0    | 0    |
| 2    | C       | 0.81 | 0.9  | 1.0  | 0    |
| 3    | B       | 0.73 | 1.81 | 0.9  | 0    |
| 4    | D       | 0.66 | 1.63 | 0.81 | 1.0  |

**Replacing traces**:

| Time | Visited | z(A) | z(B) | z(C) | z(D) |
|------|---------|------|------|------|------|
| 0    | A       | 1.0  | 0    | 0    | 0    |
| 1    | B       | 0.9  | 1.0  | 0    | 0    |
| 2    | C       | 0.81 | 0.9  | 1.0  | 0    |
| 3    | B       | 0.73 | 1.0  | 0.9  | 0    |
| 4    | D       | 0.66 | 0.9  | 0.81 | 1.0  |

## 6. TD(λ) - Forward View <a id='td-lambda-forward'></a>

### λ-Return

**Idea**: Combine all n-step returns with weights

**λ-return**:
$$G_t^\lambda = (1-\lambda) \sum_{n=1}^\infty \lambda^{n-1} G_t^{(n)}$$

**Interpretation**: Weighted average of all n-step returns
- Weight for n-step return: $(1-\lambda)\lambda^{n-1}$
- Geometric series: $\sum_{n=1}^\infty (1-\lambda)\lambda^{n-1} = 1$ ✓

### Special Cases

**λ = 0**: Only 1-step return
$$G_t^{\lambda=0} = G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$$

This is **TD(0)**!

**λ = 1**: All returns with exponentially decaying weights → Monte Carlo
$$G_t^{\lambda=1} = G_t$$

**λ ∈ (0,1)**: Intermediate methods

### TD(λ) Update (Forward View)

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^\lambda - V(S_t)]$$

**Problem**: Forward view requires complete episode (to compute all $G_t^{(n)}$)

**Solution**: Backward view with eligibility traces (next section)

### Why λ-Return?

**Benefits**:
1. Single parameter (λ) controls bias-variance
2. Smooth interpolation between TD and MC
3. Often better than any single n
4. Equivalent to eligibility trace algorithm (more efficient)

### Truncated λ-Return

For finite episodes (terminal at T):

$$G_t^\lambda = (1-\lambda) \sum_{n=1}^{T-t-1} \lambda^{n-1} G_t^{(n)} + \lambda^{T-t-1} G_t$$

Last term is full MC return weighted by $\lambda^{T-t-1}$

## 7. TD(λ) - Backward View <a id='td-lambda-backward'></a>

### Online Algorithm with Traces

**Key insight**: Eligibility traces allow online updates equivalent to forward-view λ-return

### TD(λ) Algorithm (Backward View)

```
Initialize V(s) arbitrarily for all s
Parameters: α, γ, λ

For each episode:
  Initialize z(s) = 0 for all s
  Initialize S
  
  For each step:
    Take action A, observe R, S'
    δ ← R + γV(S') - V(S)  (TD error)
    z(S) ← z(S) + 1  (or z(S) ← 1 for replacing traces)
    
    For all s:
      V(s) ← V(s) + α δ z(s)
      z(s) ← γλ z(s)
    
    S ← S'
    If S' is terminal: break
```

### Key Features

1. **Updates all states every step**: Proportional to eligibility
2. **Online**: No need to wait for episode end
3. **Efficient**: O(|S|) per step (can be sparse)
4. **Equivalent to forward view**: For appropriate parameters

### TD Error Broadcasting

**TD error at time t**:
$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

**Update for each state**:
$$V(s) \leftarrow V(s) + \alpha \delta_t z_t(s)$$

**Interpretation**: 
- TD error computed from current transition
- "Broadcast" to all states proportional to eligibility
- Recently visited states get more credit

### SARSA(λ)

Extension to action values:

```
Initialize Q(s,a), z(s,a) = 0

For each episode:
  Initialize z(s,a) = 0 for all s,a
  S, A ← initial state and action
  
  For each step:
    Take A, observe R, S'
    Choose A' from S' (ε-greedy)
    δ ← R + γQ(S',A') - Q(S,A)
    z(S,A) ← z(S,A) + 1
    
    For all s,a:
      Q(s,a) ← Q(s,a) + α δ z(s,a)
      z(s,a) ← γλ z(s,a)
    
    S ← S'; A ← A'
    If S' terminal: break
```

### Q(λ) - Off-Policy Version

**Watkins's Q(λ)**:
- Cut traces when non-greedy action taken
- Ensures off-policy convergence

```
δ ← R + γ max_a Q(S',a) - Q(S,A)
z(S,A) ← z(S,A) + 1

For all s,a:
  Q(s,a) ← Q(s,a) + α δ z(s,a)
  If A ≠ argmax_a Q(S,a):
    z(s,a) ← 0  (cut traces)
  Else:
    z(s,a) ← γλ z(s,a)
```

## 8. Choosing n and λ <a id='choosing'></a>

### Empirical Guidelines

**For n-step methods**:

**Small problems** (few states):
- n = 3-5 often optimal
- Higher n → diminishing returns

**Large problems**:
- n = 1-3 often best
- High n → too much variance

**Short episodes**:
- Can use larger n
- Approach MC (n = episode length)

**Long/continuing tasks**:
- Use smaller n
- Large n impractical

**For λ**:

**λ = 0**:
- Pure 1-step TD
- Fast learning initially
- High bias

**λ = 0.9-0.99**:
- Often optimal
- Good bias-variance trade-off
- Works well in practice

**λ = 1**:
- Monte Carlo
- No bias, high variance
- Good for short episodes

### Problem-Dependent Factors

**High stochasticity**:
- Prefer smaller n or λ
- Reduce variance

**Deterministic dynamics**:
- Can use larger n or λ
- Variance not an issue

**Poor function approximation**:
- Smaller n or λ often better
- Bootstrapping helps when estimates poor

**Good function approximation**:
- Larger n or λ acceptable
- Trust estimates more

### Relationship Between n and λ

**Rough equivalence**:
- λ ≈ 0.9 ≈ n = 5-10
- λ ≈ 0.5 ≈ n = 2-3
- λ ≈ 0.0 = n = 1

**Advantage of λ**: Averages over all n automatically

## 9. Theory and Convergence <a id='convergence'></a>

### Convergence of n-Step Methods

**Theorem 9.1** (n-Step TD Convergence):

For any fixed policy π, n-step TD converges to $V^\pi$ under:
1. Step-sizes satisfy Robbins-Monro: $\sum_t \alpha_t = \infty$, $\sum_t \alpha_t^2 < \infty$
2. All states visited infinitely often

**Proof sketch**: Extension of 1-step TD proof using n-step Bellman operator.

### Convergence of TD(λ)

**Theorem 9.2** (TD(λ) Convergence):

For tabular case with appropriate step-sizes, TD(λ) converges to $V^\pi$ for all $\lambda \in [0,1]$.

**For linear function approximation**:
- TD(λ) converges for all λ ∈ [0,1]
- Convergence point depends on λ
- All converge to "nearby" solutions

### Convergence Rate

**Empirical observations**:

**λ = 0** (TD(0)):
- Fastest per-iteration
- May require more iterations total

**λ ∈ (0,1)** (TD(λ)):
- Often fastest overall convergence
- Best bias-variance trade-off

**λ = 1** (MC):
- Slowest (high variance)
- But unbiased

**Optimal λ**: Problem-dependent, typically 0.8-0.95

### Error Bounds

**n-Step error bound**:

$$\|V_t - V^\pi\| \leq \gamma^n \|V_0 - V^\pi\| + \text{sampling error}$$

**Trade-off**:
- ↑ n: ↓ bias (smaller $\gamma^n$ factor)
- ↑ n: ↑ variance (more sampling error)

### Computational Complexity

**n-Step methods**:
- Memory: O(n) per state (store n-step trajectory)
- Computation: O(1) per update
- Delay: n steps before update

**TD(λ) with traces**:
- Memory: O(|S|) for trace vector
- Computation: O(|S|) per step (update all states)
- With sparse traces: O(|visited states|)
- No delay: online updates

**Trade-off**: TD(λ) more computation per step, but no delay and often faster convergence

---

## Summary

In this notebook, you have learned:

1. ✅ **n-Step Returns** - Bridge TD and MC continuously
2. ✅ **n-Step TD** - Multi-step temporal difference prediction
3. ✅ **n-Step SARSA** - n-step on-policy control
4. ✅ **n-Step Off-Policy** - Tree backup and importance sampling
5. ✅ **Eligibility Traces** - Credit assignment mechanism
6. ✅ **TD(λ) Forward View** - λ-return as weighted n-step returns
7. ✅ **TD(λ) Backward View** - Online algorithm with traces
8. ✅ **Parameter Selection** - Guidelines for choosing n and λ
9. ✅ **Convergence Theory** - Guarantees and error bounds

### Key Equations

**n-Step Return**:
$$G_t^{(n)} = \sum_{i=0}^{n-1} \gamma^i R_{t+i+1} + \gamma^n V(S_{t+n})$$

**λ-Return**:
$$G_t^\lambda = (1-\lambda) \sum_{n=1}^\infty \lambda^{n-1} G_t^{(n)}$$

**Eligibility Trace**:
$$z_t(s) = \gamma \lambda z_{t-1}(s) + \mathbb{1}(S_t = s)$$

**TD(λ) Update**:
$$V(s) \leftarrow V(s) + \alpha \delta_t z_t(s)$$

where $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$

### Core Insights

1. **Spectrum**: n-step methods interpolate between TD(0) and MC
2. **Trade-off**: Bias (low n) vs variance (high n)
3. **Sweet spot**: Intermediate n or λ often best
4. **Eligibility traces**: Efficient implementation of λ-returns
5. **Credit assignment**: Traces distribute credit to past states

### Practical Guidance

**Use n-step (n=3-5) when**:
- Want simple implementation
- Episodes are short
- Memory constrained

**Use TD(λ) (λ=0.9-0.95) when**:
- Want best performance
- Can afford O(|S|) computation
- Have longer episodes

**Start with**:
- n = 3 or λ = 0.9
- Tune based on problem

### Next Steps

**Lesson 5b** will implement:
- n-Step TD prediction and SARSA
- TD(λ) with eligibility traces
- SARSA(λ) and Q(λ)
- Comprehensive comparison of all variants
- Parameter sensitivity analysis

---

**Lesson 5a Complete!** 🎉

You now understand the theoretical foundations of n-step methods and eligibility traces, completing the bridge between TD and MC methods.